# 08 Multi-Agent Architectures: Supervisor Routing Pipeline

**Scenario**: Implement a stateful multi-agent system under the Supervisor Pattern using local Ollama `llama3.1:latest`.

## Step 1: Ollama Interface Setup

In [1]:
import urllib.request
import json

def query_ollama(prompt, model="llama3.1:latest"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf-8"))
            return res.get("response", "").strip()
    except Exception as e:
        return "MathSpecialist"

print("Supervisor Routing Warmup:", query_ollama("Say hello."))

Supervisor Routing Warmup: Hello! How can I assist you today?


## Step 2: Implement Supervisor & Worker Specialists

In [2]:
class MathSpecialist:
    def run_task(self, sub_task):
        print(f"  [MathSpecialist] Processing: '{sub_task}'")
        # Simple arithmetic verification
        import re
        nums = [int(n) for n in re.findall(r"\d+", sub_task)]
        result = sum(nums)
        return f"Calculation completed. Sum of {nums} is {result}."

class TextSpecialist:
    def run_task(self, sub_task):
        print(f"  [TextSpecialist] Processing: '{sub_task}'")
        return f"Formatted text payload based on query details: '{sub_task.upper()}'"

class AgentSupervisor:
    def __init__(self):
        self.math_worker = MathSpecialist()
        self.text_worker = TextSpecialist()
        
    def route_query(self, user_query):
        print(f"\n[SUPERVISOR] Ingested user query: '{user_query}'")
        
        prompt = f"""Analyze the user query and decide who to delegate this task to.
Options:
- MathSpecialist: If task requires adding numbers, calculation, or math formulas.
- TextSpecialist: If task requires formatting, summarizing, or processing words.

Query: '{user_query}'
Respond strictly with either MathSpecialist or TextSpecialist."""
        
        selected_worker = query_ollama(prompt)
        # Standardize selection
        selected_worker = "MathSpecialist" if "math" in selected_worker.lower() else "TextSpecialist"
        print(f"[SUPERVISOR] Delegating task to: {selected_worker}")
        
        if selected_worker == "MathSpecialist":
            output = self.math_worker.run_task(user_query)
        else:
            output = self.text_worker.run_task(user_query)
            
        print(f"[SUPERVISOR] Worker task output successfully retrieved:")
        print(f"  -> {output}")
        return output

supervisor = AgentSupervisor()
supervisor.route_query("Add the values 150 and 320")
print("-" * 50)
supervisor.route_query("Summarize database stability tables")


[SUPERVISOR] Ingested user query: 'Add the values 150 and 320'


[SUPERVISOR] Delegating task to: MathSpecialist
  [MathSpecialist] Processing: 'Add the values 150 and 320'
[SUPERVISOR] Worker task output successfully retrieved:
  -> Calculation completed. Sum of [150, 320] is 470.
--------------------------------------------------

[SUPERVISOR] Ingested user query: 'Summarize database stability tables'


[SUPERVISOR] Delegating task to: TextSpecialist
  [TextSpecialist] Processing: 'Summarize database stability tables'
[SUPERVISOR] Worker task output successfully retrieved:
  -> Formatted text payload based on query details: 'SUMMARIZE DATABASE STABILITY TABLES'


"Formatted text payload based on query details: 'SUMMARIZE DATABASE STABILITY TABLES'"

## Output Explanation & Complexity Analysis

- **Supervisor Routing Logic**: The central supervisor uses LLM classification tokens to dispatch task payloads to either `MathSpecialist` or `TextSpecialist`. The specialists run context-isolated python execution code blocks.
- **Time Complexity**: $O(C \cdot T \cdot d^2)$ total tokens spent across routing checks.
- **Space Complexity**: $O(\sum N_i)$ context RAM footprints.
- **Component Denotations**:
  - $C$: Number of specialized workers ($C = 2$ specialists instantiated).
  - $T$: Supervisor routing steps ($T = 1$ turn per dispatch).
  - $d$: Model hidden dimension.